In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_scheduler
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
# Cargar dataset
file_path = "dataset_transacciones_colombia.csv"  # Asegúrate de colocar la ruta correcta
df = pd.read_csv(file_path)
df.head(10)

In [ ]:
# Preprocesamiento: Mapear categorías a índices
categorias = df["Categoria"].unique()
categoria_to_idx = {cat: idx for idx, cat in enumerate(categorias)}
df["Categoria_ID"] = df["Categoria"].map(categoria_to_idx)

In [ ]:
# Dividir en conjunto de entrenamiento y prueba
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["Descripcion"].tolist(), df["Categoria_ID"].tolist(), test_size=0.2, random_state=42
)

In [ ]:
# Cargar tokenizer y modelo BETO
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class TransactionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
# Crear DataLoaders
batch_size = 32  # Tamaño óptimo para GPU
train_dataset = TransactionDataset(train_texts, train_labels, tokenizer)
val_dataset = TransactionDataset(val_texts, val_labels, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Modelo BETO
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(categorias))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Optimizador y función de pérdida
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
loss_fn = torch.nn.CrossEntropyLoss()
num_training_steps = len(train_loader) * 5  # 5 epochs
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

In [ ]:
# Entrenamiento con validación
EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        input_ids, attention_mask, labels = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())
    
    # Validación
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    accuracy = correct / total
    print(f"Epoch {epoch+1} - Accuracy: {accuracy:.4f}")

In [ ]:
# Guardar modelo entrenado
model.save_pretrained("beto_transactions_model")
tokenizer.save_pretrained("beto_transactions_model")